In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from restaurant_risk.dataset.load import load_modeling_dataset
from restaurant_risk.dataset.splits import create_primary_temporal_splits
from restaurant_risk.modeling.baselines import add_smoothed_historical_risk, training_base_rate
from restaurant_risk.evaluation.metrics import evaluate_ranking_policy, random_expected_metrics

In [2]:
project_root = Path.cwd().resolve().parent
database_path = project_root / "data" / "restaurant_risk.duckdb"

TARGET_COLUMN = "target_high_severity"

CAPACITIES = [50, 100, 250, 500]

ALPHA_CANDIDATES = [1.0, 5.0, 10.0, 20.0]

print(f"Project root: {project_root}")
print(f"Database path: {database_path}")
print(f"Database exists: {database_path.exists()}")
print(f"Target column: {TARGET_COLUMN}")
print(f"Capacities: {CAPACITIES}")
print(f"Alpha candidates: {ALPHA_CANDIDATES}")

Project root: C:\Users\User\projects\restaurant-inspection-prioritization
Database path: C:\Users\User\projects\restaurant-inspection-prioritization\data\restaurant_risk.duckdb
Database exists: True
Target column: target_high_severity
Capacities: [50, 100, 250, 500]
Alpha candidates: [1.0, 5.0, 10.0, 20.0]


## 1. Load Validated Modeling Dataset

In [3]:
df = load_modeling_dataset(database_path)

print(f"Modeling rows: {len(df):,}")
print(f"Unique restaurants: {df['camis'].nunique():,}")
print("Overall high-severity prevalence:",f"{df[TARGET_COLUMN].mean():.2%}",)

Modeling rows: 36,037
Unique restaurants: 17,561
Overall high-severity prevalence: 29.64%


## 2. Create Frozen Temporal Splits

In [4]:
train_df, validation_df, test_df = create_primary_temporal_splits(df)

In [6]:
split_summary_rows = []
for split_name, split_df in {"train": train_df, "validation": validation_df, "test": test_df}.items():
    split_summary_rows.append(
        {
            "split": split_name,
            "examples": len(split_df),
            "unique_restaurants": split_df["camis"].nunique(),
            "positive_targets": int(split_df[TARGET_COLUMN].sum()),
            "target_prevalence": split_df[TARGET_COLUMN].mean(),
            "first_cutoff": split_df["cutoff_date"].min().date(),
            "last_cutoff": split_df["cutoff_date"].max().date()
        }
    )
split_summary = pd.DataFrame(split_summary_rows)
split_summary           
            

,split,examples,unique_restaurants,positive_targets,target_prevalence,first_cutoff,last_cutoff
0,train,12865,10070,3559,0.276642,2022-01-03,2023-12-29
1,validation,16057,12991,4531,0.282182,2024-01-02,2024-12-31
2,test,6396,4607,2379,0.371951,2025-01-02,2025-12-31


## 3. Smoothed Historical High-Severity Risk

In [8]:
train_base_rate = training_base_rate(train_df)
print("Training period high severity prevalence:",f"{train_base_rate:.2%}")

Training period high severity prevalence: 27.66%


## 4. Validation Period Smoothing Selection

In [10]:
validation_scores_by_alpha = {}

for alpha in ALPHA_CANDIDATES:
    validation_scored = add_smoothed_historical_risk(train_df=train_df, data_to_score=validation_df, alpha=alpha, score_column="smoothed_historical_high_severity_risk")

    validation_scores_by_alpha[alpha] = validation_scored

    print(
        f"alpha={alpha:<4} | "
        f"min={validation_scored['smoothed_historical_high_severity_risk'].min():.4f} | "
        f"mean={validation_scored['smoothed_historical_high_severity_risk'].mean():.4f} | "
        f"max={validation_scored['smoothed_historical_high_severity_risk'].max():.4f}"
    )

alpha=1.0  | min=0.0395 | mean=0.2826 | max=0.8794
alpha=5.0  | min=0.1257 | mean=0.2799 | max=0.6383
alpha=10.0 | min=0.1729 | mean=0.2788 | max=0.5178
alpha=20.0 | min=0.2128 | mean=0.2779 | max=0.4213


In [11]:
validation_result_rows = []

for alpha, validation_scored in validation_scores_by_alpha.items():
    for capacity in CAPACITIES:
        policy_metrics = evaluate_ranking_policy(data=validation_scored, score_column="smoothed_historical_high_severity_risk", capacity=capacity, policy_name=f"smoothed_historical_risk_alpha_{alpha}")
        random_metrics = random_expected_metrics(data=validation_scored, capacity=capacity)

        validation_result_rows.append(
            {
                "alpha": alpha,
                "capacity": capacity,
                "policy_outcomes_found": policy_metrics["high_severity_outcomes_found"],
                "policy_precision_at_k": policy_metrics["precision_at_k"],
                "policy_recall_at_k": policy_metrics["recall_at_k"],
                "policy_lift_vs_random": policy_metrics["lift_vs_random"],
                "random_expected_outcomes": random_metrics["expected_high_severity_outcomes"],
                "random_expected_precision": random_metrics["expected_precision_at_k"]
            }
        )

validation_alpha_results = pd.DataFrame(validation_result_rows)
validation_alpha_results

,alpha,capacity,policy_outcomes_found,policy_precision_at_k,policy_recall_at_k,policy_lift_vs_random,random_expected_outcomes,random_expected_precision
0,1.0,50,24,0.480,0.005297,1.701028,14.109111,0.282182
1,1.0,100,54,0.540,0.011918,1.913657,28.218223,0.282182
2,1.0,250,120,0.480,0.026484,1.701028,70.545556,0.282182
3,1.0,500,249,0.498,0.054955,1.764817,141.091113,0.282182
4,5.0,50,23,0.460,0.005076,1.630152,14.109111,0.282182
5,5.0,100,53,0.530,0.011697,1.878219,28.218223,0.282182
6,5.0,250,124,0.496,0.027367,1.757729,70.545556,0.282182
7,5.0,500,244,0.488,0.053851,1.729379,141.091113,0.282182
8,10.0,50,25,0.500,0.005518,1.771905,14.109111,0.282182
9,10.0,100,53,0.530,0.011697,1.878219,28.218223,0.282182


In [17]:
validation_alpha_results[
    [
        "alpha",
        "capacity",
        "policy_outcomes_found",
        "policy_precision_at_k",
        "policy_recall_at_k",
        "policy_lift_vs_random",
        "random_expected_outcomes",
    ]
].sort_values(["capacity", "alpha"])

,alpha,capacity,policy_outcomes_found,policy_precision_at_k,policy_recall_at_k,policy_lift_vs_random,random_expected_outcomes
0,1.0,50,24,0.480,0.005297,1.701028,14.109111
4,5.0,50,23,0.460,0.005076,1.630152,14.109111
8,10.0,50,25,0.500,0.005518,1.771905,14.109111
12,20.0,50,25,0.500,0.005518,1.771905,14.109111
1,1.0,100,54,0.540,0.011918,1.913657,28.218223
5,5.0,100,53,0.530,0.011697,1.878219,28.218223
9,10.0,100,53,0.530,0.011697,1.878219,28.218223
13,20.0,100,53,0.530,0.011697,1.878219,28.218223
2,1.0,250,120,0.480,0.026484,1.701028,70.545556
6,5.0,250,124,0.496,0.027367,1.757729,70.545556


In [18]:
validation_alpha_results["incremental_outcomes_vs_random"] = (validation_alpha_results["policy_outcomes_found"]- validation_alpha_results["random_expected_outcomes"])
validation_alpha_results[
    [
        "alpha",
        "capacity",
        "policy_outcomes_found",
        "random_expected_outcomes",
        "incremental_outcomes_vs_random",
        "policy_precision_at_k",
        "policy_lift_vs_random",
    ]
].sort_values(["capacity", "alpha"])

,alpha,capacity,policy_outcomes_found,random_expected_outcomes,incremental_outcomes_vs_random,policy_precision_at_k,policy_lift_vs_random
0,1.0,50,24,14.109111,9.890889,0.480,1.701028
4,5.0,50,23,14.109111,8.890889,0.460,1.630152
8,10.0,50,25,14.109111,10.890889,0.500,1.771905
12,20.0,50,25,14.109111,10.890889,0.500,1.771905
1,1.0,100,54,28.218223,25.781777,0.540,1.913657
5,5.0,100,53,28.218223,24.781777,0.530,1.878219
9,10.0,100,53,28.218223,24.781777,0.530,1.878219
13,20.0,100,53,28.218223,24.781777,0.530,1.878219
2,1.0,250,120,70.545556,49.454444,0.480,1.701028
6,5.0,250,124,70.545556,53.454444,0.496,1.757729


## 5. Baseline Smoothing Decision

In [19]:
SELECTED_ALPHA = 5.0

print(f"Selected smoothing alpha: {SELECTED_ALPHA}")

Selected smoothing alpha: 5.0


In [21]:
selected_validation_baseline = (validation_alpha_results[validation_alpha_results["alpha"] == SELECTED_ALPHA].copy().sort_values("capacity"))
selected_validation_baseline[
    [
        "alpha",
        "capacity",
        "policy_outcomes_found",
        "random_expected_outcomes",
        "incremental_outcomes_vs_random",
        "policy_precision_at_k",
        "policy_recall_at_k",
        "policy_lift_vs_random",
    ]
]

,alpha,capacity,policy_outcomes_found,random_expected_outcomes,incremental_outcomes_vs_random,policy_precision_at_k,policy_recall_at_k,policy_lift_vs_random
4,5.0,50,23,14.109111,8.890889,0.460,0.005076,1.630152
5,5.0,100,53,28.218223,24.781777,0.530,0.011697,1.878219
6,5.0,250,124,70.545556,53.454444,0.496,0.027367,1.757729
7,5.0,500,244,141.091113,102.908887,0.488,0.053851,1.729379


## 6. Final Test Evaluation of the Frozen Historical Risk Baseline

In [22]:
test_scored = add_smoothed_historical_risk(train_df=train_df, data_to_score=test_df, alpha=SELECTED_ALPHA, score_column="smoothed_historical_high_severity_risk")
print("Test target prevalence:",f"{test_scored[TARGET_COLUMN].mean():.2%}")
print("Test score mean:",f"{test_scored['smoothed_historical_high_severity_risk'].mean():.2%}")

Test target prevalence: 37.20%
Test score mean: 31.78%


In [26]:
test_result_rows = []

for capacity in CAPACITIES:
    policy_metrics = evaluate_ranking_policy(data=test_scored, score_column="smoothed_historical_high_severity_risk", capacity=capacity, policy_name=(f"smoothed_historical_risk_alpha_{SELECTED_ALPHA}"),)
    random_metrics = random_expected_metrics(data=test_scored, capacity=capacity)
    test_result_rows.append(
        {
            "capacity": capacity,
            "policy_outcomes_found": (policy_metrics["high_severity_outcomes_found"]),
            "policy_precision_at_k": (policy_metrics["precision_at_k"]),
            "policy_recall_at_k": (policy_metrics["recall_at_k"]),
            "policy_lift_vs_random": (policy_metrics["lift_vs_random"]),
            "random_expected_outcomes": (random_metrics["expected_high_severity_outcomes"]),
            "random_expected_precision": (random_metrics["expected_precision_at_k"])
        }
    )

test_baseline_results = pd.DataFrame(test_result_rows)
test_baseline_results["incremental_outcomes_vs_random"] = (test_baseline_results["policy_outcomes_found"]- test_baseline_results["random_expected_outcomes"])
test_baseline_results

,capacity,policy_outcomes_found,policy_precision_at_k,policy_recall_at_k,policy_lift_vs_random,random_expected_outcomes,random_expected_precision,incremental_outcomes_vs_random
0,50,33,0.660,0.013871,1.774426,18.597561,0.371951,14.402439
1,100,72,0.720,0.030265,1.935738,37.195122,0.371951,34.804878
2,250,171,0.684,0.071879,1.838951,92.987805,0.371951,78.012195
3,500,324,0.648,0.136192,1.742164,185.975610,0.371951,138.024390


In [27]:
validation_comparison = selected_validation_baseline[["capacity", "policy_outcomes_found", "policy_precision_at_k", "policy_lift_vs_random", "incremental_outcomes_vs_random",]].copy()
validation_comparison["period"] = "validation_2024"
test_comparison = test_baseline_results[["capacity", "policy_outcomes_found", "policy_precision_at_k", "policy_lift_vs_random", "incremental_outcomes_vs_random"]].copy()
test_comparison["period"] = "test_2025"
baseline_temporal_comparison = pd.concat([validation_comparison, test_comparison,], ignore_index=True,)
baseline_temporal_comparison.sort_values(["capacity", "period"])

,capacity,policy_outcomes_found,policy_precision_at_k,policy_lift_vs_random,incremental_outcomes_vs_random,period
4,50,33,0.660,1.774426,14.402439,test_2025
0,50,23,0.460,1.630152,8.890889,validation_2024
5,100,72,0.720,1.935738,34.804878,test_2025
1,100,53,0.530,1.878219,24.781777,validation_2024
6,250,171,0.684,1.838951,78.012195,test_2025
2,250,124,0.496,1.757729,53.454444,validation_2024
7,500,324,0.648,1.742164,138.024390,test_2025
3,500,244,0.488,1.729379,102.908887,validation_2024
